# Employment Outcome and Skilling Impact Analysis

This notebook analyzes the PMKVY aggregate training funnel. It measures enrollment, training, assessment, certification, and reported placement outcomes by program and geography. The source is aggregated rather than individual-level, so this pipeline prioritizes transparent impact analytics over an unsupported individual prediction model.

## Data loading and inspection

The schema is checked explicitly. Paths are resolved with `pathlib` so the notebook can run from the project root or its notebook directory.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'ml' / 'data').exists())
DATA_PATH = PROJECT_ROOT / 'ml' / 'data' / 'PMKVY-210422.csv'
MODEL_DIR = PROJECT_ROOT / 'ml' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), DATA_PATH
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
print('columns:', df.columns.tolist())
display(df.head())
display(df.sample(min(5, len(df)), random_state=42))
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing'))
print('duplicate rows:', int(df.duplicated().sum()))
display(df.describe(include='all').T)

## Cleaning and data-quality checks

The dataset contains one reporting date, so time-series forecasting is not supported. Counts are converted explicitly and impossible relationships are flagged rather than silently removed. A duplicate natural key is retained in the source but excluded from grouped totals to avoid double-counting.

In [ ]:
measure_columns = ['Enrolled', 'Trained', 'Assessed', 'Certified', 'Reported Placed']
dimension_columns = ['Data Till', 'Scheme', 'Component', 'TrainingType', 'TCState', 'TCDistrict']
required_columns = dimension_columns + measure_columns
missing_columns = sorted(set(required_columns) - set(df.columns))
assert not missing_columns, missing_columns
for column in measure_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['Data Till'] = pd.to_datetime(df['Data Till'], dayfirst=True, errors='coerce')
invalid_numeric = df[measure_columns].isna().any(axis=1)
natural_key = dimension_columns
duplicate_key = df.duplicated(natural_key, keep='first')
df['quality_flag'] = False
df['quality_flag'] |= invalid_numeric
df['quality_flag'] |= (df[measure_columns] < 0).any(axis=1)
df['quality_flag'] |= df['Trained'].gt(df['Enrolled'])
df['quality_flag'] |= df['Assessed'].gt(df['Trained'])
df['quality_flag'] |= df['Certified'].gt(df['Assessed'])
df['quality_flag'] |= df['Reported Placed'].gt(df['Certified'])
quality_summary = {
    'invalid_numeric_rows': int(invalid_numeric.sum()),
    'duplicate_natural_keys': int(duplicate_key.sum()),
    'negative_measure_rows': int((df[measure_columns] < 0).any(axis=1).sum()),
    'trained_gt_enrolled': int(df['Trained'].gt(df['Enrolled']).sum()),
    'assessed_gt_trained': int(df['Assessed'].gt(df['Trained']).sum()),
    'certified_gt_assessed': int(df['Certified'].gt(df['Assessed']).sum()),
    'placed_gt_certified': int(df['Reported Placed'].gt(df['Certified']).sum()),
}
print(json.dumps(quality_summary, indent=2))
print('flagged rows:', int(df['quality_flag'].sum()))

In [ ]:
# Aggregate duplicate natural keys conservatively for reporting totals.
analysis_df = (df.groupby(natural_key, as_index=False, dropna=False)[measure_columns].sum())
def safe_rate(numerator, denominator):
    return np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=float), where=denominator.ne(0))

analysis_df['training_completion_rate'] = safe_rate(analysis_df['Trained'], analysis_df['Enrolled'])
analysis_df['assessment_rate'] = safe_rate(analysis_df['Assessed'], analysis_df['Trained'])
analysis_df['certification_rate'] = safe_rate(analysis_df['Certified'], analysis_df['Assessed'])
analysis_df['placement_rate'] = safe_rate(analysis_df['Reported Placed'], analysis_df['Certified'])
analysis_df['pipeline_dropoff_rate'] = 1 - safe_rate(analysis_df['Reported Placed'], analysis_df['Enrolled'])
analysis_df['program_effectiveness_score'] = (
    analysis_df['training_completion_rate'] * analysis_df['assessment_rate'] *
    analysis_df['certification_rate'] * analysis_df['placement_rate']
)
display(analysis_df[measure_columns + ['training_completion_rate', 'assessment_rate', 'certification_rate', 'placement_rate', 'pipeline_dropoff_rate']].describe().T)
display(analysis_df.sort_values('Reported Placed', ascending=False).head(10))

## Performance views

These views compare groups only where the relevant denominator exists. They are descriptive and should not be interpreted as causal estimates.

In [ ]:
def grouped_performance(group_columns):
    grouped = analysis_df.groupby(group_columns, as_index=False)[measure_columns].sum()
    grouped['training_completion_rate'] = safe_rate(grouped['Trained'], grouped['Enrolled'])
    grouped['assessment_rate'] = safe_rate(grouped['Assessed'], grouped['Trained'])
    grouped['certification_rate'] = safe_rate(grouped['Certified'], grouped['Assessed'])
    grouped['placement_rate'] = safe_rate(grouped['Reported Placed'], grouped['Certified'])
    grouped['pipeline_dropoff_rate'] = 1 - safe_rate(grouped['Reported Placed'], grouped['Enrolled'])
    return grouped

for grouping in [['Scheme'], ['Component'], ['TrainingType'], ['TCState']]:
    view = grouped_performance(grouping).sort_values('Reported Placed', ascending=False)
    print('\n', grouping)
    display(view.head(15))

state_view = grouped_performance(['TCState']).sort_values('placement_rate', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=state_view.head(10), y='TCState', x='placement_rate', color='#2a9d8f')
plt.title('Top states by reported placement rate')
plt.xlabel('Reported placed / certified')
plt.ylabel('')
plt.tight_layout()
plt.show()

training_view = grouped_performance(['TrainingType']).sort_values('Reported Placed', ascending=False)
training_view.set_index('TrainingType')[['training_completion_rate', 'assessment_rate', 'certification_rate', 'placement_rate']].plot(kind='bar', figsize=(10, 5))
plt.ylim(0, 1.05)
plt.title('Training pipeline rates by training type')
plt.ylabel('Rate')
plt.tight_layout()
plt.show()

## Reusable employment artifact

Because the source has one date and aggregate rows, the saved artifact is an analytics configuration and summary rather than a fabricated predictive model. It contains the schema, funnel definitions, quality findings, and grouped summaries needed by a future engine.

In [ ]:
artifact = {
    'artifact_type': 'pmkvy_employment_analytics',
    'version': 1,
    'source_file': DATA_PATH.name,
    'row_count': int(len(df)),
    'analysis_row_count': int(len(analysis_df)),
    'columns': required_columns,
    'natural_key': natural_key,
    'measure_columns': measure_columns,
    'quality_summary': quality_summary,
    'metric_definitions': {
        'training_completion_rate': 'Trained / Enrolled when Enrolled > 0',
        'assessment_rate': 'Assessed / Trained when Trained > 0',
        'certification_rate': 'Certified / Assessed when Assessed > 0',
        'placement_rate': 'Reported Placed / Certified when Certified > 0',
        'pipeline_dropoff_rate': '1 - Reported Placed / Enrolled when Enrolled > 0',
        'program_effectiveness_score': 'Product of the four stage rates',
    },
    'grouped_summaries': {
        'scheme': grouped_performance(['Scheme']).to_dict('records'),
        'component': grouped_performance(['Component']).to_dict('records'),
        'training_type': grouped_performance(['TrainingType']).to_dict('records'),
        'state': grouped_performance(['TCState']).to_dict('records'),
    },
    'modeling_decision': 'analytics_only: one reporting date and aggregate program/geography grain',
}
artifact_path = MODEL_DIR / 'employment_model.joblib'
joblib.dump(artifact, artifact_path)
assert artifact_path.exists() and artifact_path.stat().st_size > 0
loaded_artifact = joblib.load(artifact_path)
assert loaded_artifact['artifact_type'] == 'pmkvy_employment_analytics'
assert loaded_artifact['quality_summary'] == quality_summary
print('saved:', artifact_path)
print('artifact keys:', sorted(loaded_artifact))